# sign1_updated_v3

Builds MP_Data keypoints (1662 per frame) from BU ASLLVD Excel + scene videos.

Output:
`MP_Data/<word>/<sequence>/<frame>.npy`


In [ ]:
# --- Imports ---
import os
import re
import time
import requests
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp


In [ ]:
# --- Paths (EDIT THESE) ---
WORDS_TXT   = r"C:\Users\praya\Desktop\VSCODE\AI_for_MAJOR_PROJECT\words.txt"
XLSX_PATH   = r"C:\Users\praya\Desktop\VSCODE\AI_for_MAJOR_PROJECT\asllvd.xlsx"
MP_DATA     = r"C:\Users\praya\Desktop\VSCODE\AI_for_MAJOR_PROJECT\MP_Data"
SCENE_CACHE = r"C:\Users\praya\Desktop\VSCODE\AI_for_MAJOR_PROJECT\SCENE_VIDEOS"

os.makedirs(MP_DATA, exist_ok=True)
os.makedirs(SCENE_CACHE, exist_ok=True)


In [ ]:
# --- Settings ---
CAMERA = 1
SEQUENCE_LENGTH = 30   # frames per sample
MAX_PER_WORD = 30      # clips per word (sequence folders 0..29)

BASE_QT = "http://csr.bu.edu/ftp/asl/asllvd/asl-data2/quicktime/{session}/scene{scene}-camera{camera}.mov"


In [ ]:
# --- Load your 200 words from words.txt ---
with open(WORDS_TXT, "r", encoding="utf-8") as f:
    actions = [line.strip() for line in f if line.strip()]

wanted = set(actions)
print("Words loaded:", len(actions))
print("Sample:", actions[:10])


In [ ]:
# --- Helpers: cleaning + resumable download (416-safe) ---
def clean_gloss(x) -> str:
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    # safe folder names on Windows
    s = re.sub(r'[<>:"/\\|?*]', "", s)
    return s

def download_file_resumable(url, out_path, max_retries=8, chunk_size=1024*1024):
    """Resumable downloader with HTTP 416 fix."""
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    def head_size():
        try:
            h = requests.head(url, timeout=20, allow_redirects=True,
                              headers={"User-Agent": "Mozilla/5.0"})
            if "Content-Length" in h.headers:
                return int(h.headers["Content-Length"])
        except:
            return None
        return None

    total_size = head_size()

    for attempt in range(1, max_retries + 1):
        existing = os.path.getsize(out_path) if os.path.exists(out_path) else 0

        if total_size is not None and existing >= total_size and total_size > 0:
            return True

        headers = {"User-Agent": "Mozilla/5.0"}
        if existing > 0:
            headers["Range"] = f"bytes={existing}-"

        try:
            r = requests.get(url, stream=True, timeout=(15, 180),
                             headers=headers, allow_redirects=True)

            if r.status_code == 416:
                try:
                    r.close()
                except:
                    pass
                if os.path.exists(out_path):
                    os.remove(out_path)
                time.sleep(min(10, 2 ** attempt))
                continue

            if r.status_code not in (200, 206):
                print(f"[HTTP {r.status_code}] {url}")
                return False

            mode = "ab" if (existing > 0 and r.status_code == 206) else "wb"
            with open(out_path, mode) as f:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if chunk:
                        f.write(chunk)

            if total_size is None:
                total_size = head_size()

            size_now = os.path.getsize(out_path)
            if size_now > 0 and (total_size is None or size_now >= total_size):
                return True

        except Exception as e:
            wait = min(60, 2 ** attempt)
            print(f"[Retry {attempt}/{max_retries}] {type(e).__name__}: {e}")
            time.sleep(wait)

    return False

def get_segment_indices(start_f, end_f, n=30):
    start_f = int(start_f)
    end_f = int(end_f)
    if end_f <= start_f:
        return [start_f] * n
    return np.linspace(start_f, end_f, n).astype(int).tolist()


In [ ]:
# --- Load and clean the ASLLVD Excel ---
df = pd.read_excel(XLSX_PATH, engine="openpyxl")

gloss_raw = df["Main New Gloss.1"].where(df["Main New Gloss.1"].notna(), df["Main New Gloss"])
df["gloss"] = gloss_raw.apply(lambda x: clean_gloss(x) if pd.notna(x) else np.nan)

for c in ["Scene", "Start", "End"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df2 = df.dropna(subset=["gloss", "Session", "Scene", "Start", "End"]).copy()
df2["Session"] = df2["Session"].astype(str).str.strip()
df2["Scene"] = df2["Scene"].astype(int)
df2["Start"] = df2["Start"].astype(int)
df2["End"] = df2["End"].astype(int)
df2 = df2[df2["End"] > df2["Start"]]

print("Usable rows after cleaning:", len(df2))
df2[["gloss","Session","Scene","Start","End"]].head(10)


In [ ]:
# --- Filter to your 200 words + cap to 30 clips per word ---
df2 = df2[df2["gloss"].isin(wanted)].copy()
df2 = df2.sort_values(["gloss", "Session", "Scene", "Start"])
df2 = df2.groupby("gloss").head(MAX_PER_WORD).reset_index(drop=True)

print("Rows after filter+cap:", len(df2))
print("Unique words found in ASLLVD:", df2["gloss"].nunique())
df2["gloss"].value_counts().head(20)


In [ ]:
# --- MediaPipe Holistic: 1662 keypoints per frame ---
mp_holistic = mp.solutions.holistic

def mediapipe_results(frame, model):
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img.flags.writeable = False
    results = model.process(img)
    return results

def extract_keypoints(results):
    pose = np.array([[lm.x, lm.y, lm.z, lm.visibility]
                     for lm in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)

    face = np.array([[lm.x, lm.y, lm.z]
                     for lm in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)

    lh = np.array([[lm.x, lm.y, lm.z]
                   for lm in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)

    rh = np.array([[lm.x, lm.y, lm.z]
                   for lm in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)

    return np.concatenate([pose, face, lh, rh]).astype(np.float32)


In [ ]:
# --- Process: download scene once (cached) -> extract segment -> save 30 npy files ---
downloaded_scene_keys = set()
seq_counter = {w: 0 for w in actions}

processed = 0
skipped = 0

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:

    for _, row in df2.iterrows():
        gloss = row["gloss"]
        session = row["Session"]
        scene = int(row["Scene"])
        start_f = int(row["Start"])
        end_f = int(row["End"])

        sequence = seq_counter.get(gloss, 0)
        if sequence >= MAX_PER_WORD:
            skipped += 1
            continue
        seq_counter[gloss] = sequence + 1

        url = BASE_QT.format(session=session, scene=scene, camera=CAMERA)
        scene_key = f"{session}_scene{scene}_cam{CAMERA}"
        scene_path = os.path.join(SCENE_CACHE, f"{scene_key}.mov")

        if scene_key not in downloaded_scene_keys:
            if not (os.path.exists(scene_path) and os.path.getsize(scene_path) > 1_000_000):
                ok = download_file_resumable(url, scene_path)
                if not ok:
                    print("Download failed:", url)
                    skipped += 1
                    continue
            downloaded_scene_keys.add(scene_key)

        cap = cv2.VideoCapture(scene_path)
        if not cap.isOpened():
            print("OpenCV cannot open:", scene_path)
            skipped += 1
            continue

        frame_indices = get_segment_indices(start_f, end_f, SEQUENCE_LENGTH)
        out_dir = os.path.join(MP_DATA, gloss, str(sequence))
        os.makedirs(out_dir, exist_ok=True)

        for frame_num, frame_idx in enumerate(frame_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            ret, frame = cap.read()

            if not ret:
                keypoints = np.zeros(1662, dtype=np.float32)
            else:
                results = mediapipe_results(frame, holistic)
                keypoints = extract_keypoints(results)

            np.save(os.path.join(out_dir, f"{frame_num}.npy"), keypoints)

        cap.release()
        processed += 1

        if processed % 25 == 0:
            print(f"Progress: {processed}/{len(df2)} processed | skipped: {skipped}")

print("Done ✅")
print("Processed:", processed)
print("Skipped:", skipped)


In [ ]:
# --- Sanity check (1662,) ---
test_word = next((w for w in actions if os.path.isdir(os.path.join(MP_DATA, w))), None)

if test_word:
    test_path = os.path.join(MP_DATA, test_word, "0", "0.npy")
    if os.path.exists(test_path):
        arr = np.load(test_path)
        print("Test word:", test_word)
        print("Numpy shape:", arr.shape)  # (1662,)
    else:
        print("No saved npy found yet at:", test_path)
else:
    print("No word folders created yet in MP_Data.")


## Training (Robust to missing words / incomplete sequences)

This section builds `X` and `y` from the saved `.npy` files inside `DATA_PATH` and trains an LSTM.
It **auto-detects which words actually have usable data** and **skips incomplete sequences** so the notebook doesn’t crash.

In [ ]:
# === PREPROCESS DATA + CREATE LABELS/FEATURES (ROBUST) ===
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import numpy as np
import os

# IMPORTANT: In this notebook, DATA_PATH is where your numpy data is stored (MP_Data)
print("DATA_PATH:", DATA_PATH)
print("Requested actions from words.txt:", len(actions))

# Settings for robustness
EXPECTED_LEN = 1662
MIN_SEQS_PER_ACTION = 3          # drop words with fewer than this many usable sequences
ALLOW_PAD_MISSING_FRAMES = True  # if True, pad missing frames with zeros
MAX_MISSING_FRAMES = 2           # only pad if <= this many frames missing in a sequence

def safe_load_vec(path, expected_len=1662):
    vec = np.load(path).astype(np.float32).reshape(-1)
    if vec.shape[0] < expected_len:
        vec = np.pad(vec, (0, expected_len - vec.shape[0]))
    elif vec.shape[0] > expected_len:
        vec = vec[:expected_len]
    return vec

def list_sequence_dirs(action_dir):
    if not os.path.isdir(action_dir):
        return []
    dirs = [d for d in os.listdir(action_dir) if d.isdigit() and os.path.isdir(os.path.join(action_dir, d))]
    return sorted(dirs, key=lambda x: int(x))

def load_sequence(seq_dir, sequence_length=30):
    """Return (sequence_length, EXPECTED_LEN) or None if unusable."""
    present = []
    missing = 0
    for i in range(sequence_length):
        p = os.path.join(seq_dir, f"{i}.npy")
        if os.path.exists(p):
            present.append(p)
        else:
            present.append(None)
            missing += 1

    if missing == 0:
        return np.stack([safe_load_vec(p, EXPECTED_LEN) for p in present], axis=0)

    # If missing frames, optionally pad with zeros
    if ALLOW_PAD_MISSING_FRAMES and missing <= MAX_MISSING_FRAMES:
        frames = []
        for p in present:
            if p is None:
                frames.append(np.zeros(EXPECTED_LEN, dtype=np.float32))
            else:
                frames.append(safe_load_vec(p, EXPECTED_LEN))
        return np.stack(frames, axis=0)

    return None

# Discover which actions exist on disk and have at least MIN_SEQS_PER_ACTION usable sequences
disk_actions = sorted([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))])
requested = list(actions)  # actions loaded earlier from words.txt
candidate_actions = [a for a in requested if a in disk_actions]

usable_actions = []
usable_counts = {}

for a in candidate_actions:
    a_dir = os.path.join(DATA_PATH, a)
    seq_dirs = list_sequence_dirs(a_dir)
    usable = 0
    for s in seq_dirs:
        seq = load_sequence(os.path.join(a_dir, s), sequence_length=SEQUENCE_LENGTH)
        if seq is not None:
            usable += 1
    usable_counts[a] = usable
    if usable >= MIN_SEQS_PER_ACTION:
        usable_actions.append(a)

print("Actions on disk:", len(disk_actions))
print("Requested actions found on disk:", len(candidate_actions))
print("Usable actions kept:", len(usable_actions))
print("Lowest usable counts:", sorted(usable_counts.items(), key=lambda x: x[1])[:15])

# Build label map only from usable actions
usable_actions = np.array(sorted(usable_actions))
label_map = {label: num for num, label in enumerate(usable_actions)}

sequences, labels = [], []
skipped_sequences = 0

for action in usable_actions:
    a_dir = os.path.join(DATA_PATH, action)
    seq_dirs = list_sequence_dirs(a_dir)

    for s in seq_dirs:
        seq_arr = load_sequence(os.path.join(a_dir, s), sequence_length=SEQUENCE_LENGTH)
        if seq_arr is None:
            skipped_sequences += 1
            continue
        sequences.append(seq_arr)
        labels.append(label_map[action])

X = np.array(sequences, dtype=np.float32)
y = to_categorical(labels, num_classes=len(usable_actions)).astype(np.int32)

print("X:", X.shape, "y:", y.shape)
print("Skipped sequences:", skipped_sequences)

# Stratified split if possible
if len(set(labels)) > 1 and len(labels) >= 10:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.10, random_state=42, stratify=labels
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.10, random_state=42
    )

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

In [ ]:
# === BUILD + TRAIN LSTM MODEL (UPDATED FOR USABLE_ACTIONS) ===
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping, ModelCheckpoint
import os

num_classes = len(usable_actions)
print("Classes:", num_classes)

log_dir = os.path.join("Logs")
os.makedirs(log_dir, exist_ok=True)
tb_callback = TensorBoard(log_dir=log_dir)

ckpt_path = os.path.join("models", "asl_lstm_best.keras")
os.makedirs("models", exist_ok=True)
ckpt = ModelCheckpoint(ckpt_path, monitor="val_categorical_accuracy", save_best_only=True, verbose=1)

early = EarlyStopping(monitor="val_categorical_accuracy", patience=10, restore_best_weights=True)

model = Sequential([
    LSTM(64, return_sequences=True, activation="relu", input_shape=(SEQUENCE_LENGTH, 1662)),
    Dropout(0.2),
    LSTM(128, return_sequences=True, activation="relu"),
    Dropout(0.2),
    LSTM(64, return_sequences=False, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["categorical_accuracy"])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=80,
    batch_size=16,
    callbacks=[tb_callback, early, ckpt]
)

# Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", loss)
print("Test categorical_accuracy:", acc)
print("Saved best model to:", ckpt_path)